<a href="https://colab.research.google.com/github/chrishg23-jpg/HES-benchmark/blob/main/Fractal009n100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# --- 1. CORE HES PARAMETERS ---
N = 100  # THE CRITICAL DIMENSIONAL HARMONIC
T_STEPS = 5000
dt = 0.01

# Fixed HES coefficients (final, stable values)
chi = 1.5
beta_full = 0.8  # Full Gravitational Drive strength
beta_nursery = 0.01  # Nursery Phase strength (for t < 500)
delta = 0.001  # Quantum Noise
gamma = 1.0  # Saturation
beta_link_base = 0.1
CURVATURE_SENSITIVITY = 0.5 # Ultimate Resilience Factor
MAX_CURVATURE_CAP = 50.0 # ACT X: NEW CAP to prevent numerical overflow

# --- 2. INITIAL FIELD: THREE LINKED SPINORS ---
# We initialize the field complex (Phi_complex) directly to manage both magnitude and phase
Phi_complex = np.zeros((N, N), dtype=complex)

# Initialize three spinor knots at separate locations
def initialize_spinor(complex_field, center_x, center_y, radius=10, magnitude=5.0):
    for i in range(N):
        for j in range(N):
            if (i - center_x)**2 + (j - center_y)**2 < radius**2:
                # 4pi winding for phase topology
                phase = np.arctan2(i - center_x, j - center_y) * 4
                complex_field[i, j] = magnitude * np.exp(1j * phase)

initialize_spinor(Phi_complex, N // 4, N // 4)
initialize_spinor(Phi_complex, N // 2, N // 2)
initialize_spinor(Phi_complex, 3 * N // 4, 3 * N // 4)

# Separate magnitude and phase for easier evolution
Phi = np.abs(Phi_complex)
Theta = np.angle(Phi_complex)

# --- 3. THE EVOLUTION LOOP ---
for t in range(1, T_STEPS + 1):

    # ACT VI: LAW OF PRE-STABILIZATION (Cosmic Nursery)
    beta_current = beta_nursery if t < 500 else beta_full

    # --- 3.1. CALCULATE BASE GRAVITATIONAL DRIVE (CURVATURE) ---
    lap_xy = (np.roll(Phi, 1, 0) + np.roll(Phi, -1, 0) +
              np.roll(Phi, 1, 1) + np.roll(Phi, -1, 1) - 4 * Phi) / (2 * np.pi / N)**2 # Normalized Laplacian

    # --- 3.2. IMPLEMENT PHASE-LINK RESILIENCE (ACT X - Bounded Resistance) ---
    # Max threat is observed, but CAPPED before calculating the defensive response
    max_abs_curvature_observed = np.max(np.abs(lap_xy))

    # The Bounded Resistance Law: Curvature used for defense is capped
    max_abs_curvature_capped = np.clip(max_abs_curvature_observed, 0.0, MAX_CURVATURE_CAP)

    beta_link = beta_link_base + CURVATURE_SENSITIVITY * max_abs_curvature_capped

    # --- 3.3. MAGNITUDE (PHI) EVOLUTION ---
    alpha = chi * beta_current

    # Term 1: Net Expansion
    term_expansion = alpha * lap_xy

    # Term 2: Contraction (Graviton Drive)
    term_contraction = -beta_current * Phi

    # Term 3: Saturation
    term_saturation = gamma * np.tanh(Phi)

    # Term 5: CRITICAL SHIELDING PROTOCOL (Local Resistance)
    # This shields the field magnitude ONLY where the resilience is high
    shield_mask = (beta_link > 1.0)
    term_shield = np.where(shield_mask, beta_current * Phi, 0.0)

    # Total Change dPhi/dt
    dPhi = term_expansion + term_contraction + term_saturation + term_shield + delta * np.random.normal(0, 1, Phi.shape)

    # --- 3.4. PHASE (THETA) EVOLUTION (ACT VIII: COHERENCE DRIVE) ---
    mean_theta = np.mean(Theta) # Global mean phase

    # Phase Correction Term: drives phase towards the global mean (coherence)
    phase_correction = -beta_link * (Theta - mean_theta)

    # Phase Diffusion (random phase drift)
    phase_diffusion = delta * np.random.normal(0, 1, Theta.shape)

    dTheta = phase_correction + phase_diffusion

    # --- 3.5. UPDATE FIELDS ---
    Phi += dt * dPhi
    Theta += dt * dTheta

    # Apply clipping and normalization
    Phi = np.clip(Phi, 0.01, 10.0) # Magnitude clip
    Theta = np.mod(Theta, 2 * np.pi) # Phase normalization (0 to 2pi range)

    # --- 3.6. LOGGING ---
    if t % 500 == 0:
        norm = np.mean(Phi)
        # Use a stable calculation for phase Stdev
        phase_stdev = np.sqrt(np.mean((Theta - np.mean(Theta))**2))

        # Check for annihilation (Norm < 0.1)
        if norm < 0.1:
            print(f"t={t} | ANNIHILATION DETECTED: Norm={norm:.3f}")
            break
        print(f"t={t} | β_link(t)={beta_link:.4f} | MAX Curvature={max_abs_curvature_observed:.3f} | Phase Stdev={phase_stdev:.4f} | Norm={norm:.3f}")

# --- 4. CONCLUSION CHECK ---
final_phase_stdev = np.sqrt(np.mean((Theta - np.mean(Theta))**2))
final_norm = np.mean(Phi)

print("\n--- FINAL STATE ---")
if final_phase_stdev < 0.1 and final_norm > 0.5:
    print(f"RESULT: SUCCESS. Persistence Law Validated.")
    print(f"FINAL Metrics: Phase Stdev={final_phase_stdev:.4f}, Mean Norm={final_norm:.4f}")
    print("CONCLUSION: The stable emergence and coexistence of multiple spinors on the N=100 harmonic is confirmed.")
else:
    print(f"RESULT: FAILURE. Persistence Law Falsified.")
    print(f"FINAL Metrics: Phase Stdev={final_phase_stdev:.4f}, Mean Norm={final_norm:.4f}")
    print("CONCLUSION: Multiple matter structures could not coexist autonomously.")


t=500 | β_link(t)=14.3406 | MAX Curvature=28.481 | Phase Stdev=0.0000 | Norm=2.701
t=1000 | β_link(t)=25.1000 | MAX Curvature=10121.986 | Phase Stdev=0.0000 | Norm=4.794
t=1500 | β_link(t)=25.1000 | MAX Curvature=10121.986 | Phase Stdev=0.0000 | Norm=4.794
t=2000 | β_link(t)=25.1000 | MAX Curvature=10121.986 | Phase Stdev=0.0000 | Norm=4.794
t=2500 | β_link(t)=25.1000 | MAX Curvature=10121.986 | Phase Stdev=0.0000 | Norm=4.794
t=3000 | β_link(t)=25.1000 | MAX Curvature=10121.986 | Phase Stdev=0.0000 | Norm=4.794
t=3500 | β_link(t)=25.1000 | MAX Curvature=10121.986 | Phase Stdev=0.0000 | Norm=4.794
t=4000 | β_link(t)=25.1000 | MAX Curvature=10121.986 | Phase Stdev=0.0000 | Norm=4.794
t=4500 | β_link(t)=25.1000 | MAX Curvature=10121.986 | Phase Stdev=0.0000 | Norm=4.794
t=5000 | β_link(t)=25.1000 | MAX Curvature=10121.986 | Phase Stdev=0.0000 | Norm=4.794

--- FINAL STATE ---
RESULT: SUCCESS. Persistence Law Validated.
FINAL Metrics: Phase Stdev=0.0000, Mean Norm=4.7942
CONCLUSION: The s